# 🎯 Liveness Detection Training (Clean Version)

A clean, step-by-step notebook to train a liveness detection model.

**Run cells in order from top to bottom.**

## Cell 1: Setup and Imports

In [1]:
# ============================================
# CELL 1: SETUP AND IMPORTS
# ============================================
import os
import cv2
import numpy as np
import random
from pathlib import Path
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split, Subset, ConcatDataset
from torchvision import transforms, models

# Set random seeds for reproducibility
random.seed(42)
np.random.seed(42)
torch.manual_seed(42)

# Set device
if torch.backends.mps.is_available():
    device = torch.device('mps')
    print("✅ Using Apple Silicon GPU (MPS)")
elif torch.cuda.is_available():
    device = torch.device('cuda')
    print("✅ Using NVIDIA GPU (CUDA)")
else:
    device = torch.device('cpu')
    print("⚠️ Using CPU")

# Paths
DATA_DIR = '../data'
MODELS_DIR = '../models'
OULU_PATH = os.path.join(DATA_DIR, 'oulu-npu')
TAPAKAH_PATH = os.path.join(DATA_DIR, 'anti-spoofing')

os.makedirs(MODELS_DIR, exist_ok=True)

# Check datasets
print(f"\n{'='*50}")
print("DATASET CHECK")
print(f"{'='*50}")
print(f"OULU-NPU: {'✅ Found' if os.path.exists(OULU_PATH) else '❌ Not found'}")
print(f"Tapakah68: {'✅ Found' if os.path.exists(TAPAKAH_PATH) else '❌ Not found'}")
print(f"{'='*50}")

✅ Using Apple Silicon GPU (MPS)

DATASET CHECK
OULU-NPU: ✅ Found
Tapakah68: ✅ Found


## Cell 2: Dataset Class

In [2]:
# ============================================
# CELL 2: DATASET CLASS
# ============================================

class LivenessDataset(Dataset):
    """Dataset for liveness detection - handles images and videos."""
    
    def __init__(self, root_dir, transform=None):
        self.root_dir = Path(root_dir)
        self.transform = transform
        self.samples = []  # (path, label, is_video)
        
        # Real folders (label=1)
        real_dirs = ['true', 'True', 'live_selfie', 'live_video']
        # Spoof folders (label=0)
        spoof_dirs = ['false', 'False', 'printouts', 'replay', 'cut-out printouts']
        
        for dir_name in real_dirs:
            self._load_from_dir(dir_name, label=1)
        for dir_name in spoof_dirs:
            self._load_from_dir(dir_name, label=0)
        
        real_count = sum(1 for _, l, _ in self.samples if l == 1)
        spoof_count = len(self.samples) - real_count
        print(f"Loaded {len(self.samples)} samples: {real_count} real, {spoof_count} spoof")
    
    def _load_from_dir(self, dir_name, label):
        for path in self.root_dir.rglob(dir_name):
            if path.is_dir():
                for f in path.glob('*'):
                    ext = f.suffix.lower()
                    if ext in ['.jpg', '.jpeg', '.png', '.bmp']:
                        self.samples.append((str(f), label, False))
                    elif ext in ['.mp4', '.avi', '.mov']:
                        self.samples.append((str(f), label, True))
    
    def _load_image(self, path):
        img = cv2.imread(path)
        if img is None:
            return np.zeros((224, 224, 3), dtype=np.uint8)
        return cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    
    def _load_video_frame(self, path):
        cap = cv2.VideoCapture(path)
        if not cap.isOpened():
            return np.zeros((224, 224, 3), dtype=np.uint8)
        total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        if total <= 0:
            cap.release()
            return np.zeros((224, 224, 3), dtype=np.uint8)
        cap.set(cv2.CAP_PROP_POS_FRAMES, random.randint(0, total-1))
        ret, frame = cap.read()
        cap.release()
        if not ret:
            return np.zeros((224, 224, 3), dtype=np.uint8)
        return cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    
    def __len__(self):
        return len(self.samples)
    
    def __getitem__(self, idx):
        path, label, is_video = self.samples[idx]
        img = self._load_video_frame(path) if is_video else self._load_image(path)
        if self.transform:
            img = self.transform(img)
        return img, label

print("✅ LivenessDataset class defined")

✅ LivenessDataset class defined


## Cell 3: Model Architecture

In [3]:
# ============================================
# CELL 3: MODEL ARCHITECTURE
# ============================================

class LivenessDetector(nn.Module):
    """MobileNetV2-based liveness detector."""
    
    def __init__(self, pretrained=True):
        super().__init__()
        self.backbone = models.mobilenet_v2(weights='IMAGENET1K_V1' if pretrained else None)
        in_features = self.backbone.classifier[1].in_features
        self.backbone.classifier = nn.Sequential(
            nn.Dropout(0.5),
            nn.Linear(in_features, 256),
            nn.ReLU(inplace=True),
            nn.Dropout(0.25),
            nn.Linear(256, 2)  # [spoof, real]
        )
    
    def forward(self, x):
        return self.backbone(x)
    
    def freeze_backbone(self):
        for name, param in self.backbone.named_parameters():
            if 'classifier' not in name:
                param.requires_grad = False
    
    def unfreeze_backbone(self):
        for param in self.backbone.parameters():
            param.requires_grad = True

print("✅ LivenessDetector class defined")

✅ LivenessDetector class defined


## Cell 4: Transforms and Data Loading

In [4]:
# ============================================
# CELL 4: TRANSFORMS AND DATA LOADING
# ============================================

# Transforms
train_transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(0.5),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3),
    transforms.RandomRotation(15),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

val_transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

# Load datasets
print("Loading OULU-NPU dataset...")
oulu_dataset = LivenessDataset(OULU_PATH, transform=train_transform)

print("\nLoading Tapakah68 dataset...")
tapakah_dataset = LivenessDataset(TAPAKAH_PATH, transform=train_transform)

print(f"\n{'='*50}")
print("DATASETS LOADED")
print(f"{'='*50}")

Loading OULU-NPU dataset...
Loaded 3402 samples: 686 real, 2716 spoof

Loading Tapakah68 dataset...
Loaded 45 samples: 18 real, 27 spoof

DATASETS LOADED


## Cell 5: Create Balanced Combined Dataset

In [5]:
# ============================================
# CELL 5: CREATE BALANCED COMBINED DATASET
# ============================================

# Get indices for real and spoof from OULU
oulu_real_idx = [i for i, (_, l, _) in enumerate(oulu_dataset.samples) if l == 1]
oulu_spoof_idx = [i for i, (_, l, _) in enumerate(oulu_dataset.samples) if l == 0]

# Balance OULU: undersample spoof to match real
n_real = len(oulu_real_idx)
balanced_spoof_idx = random.sample(oulu_spoof_idx, min(n_real, len(oulu_spoof_idx)))
balanced_oulu_idx = oulu_real_idx + balanced_spoof_idx
random.shuffle(balanced_oulu_idx)

balanced_oulu = Subset(oulu_dataset, balanced_oulu_idx)

# Combine with Tapakah68 (which has phone replay attacks)
combined_dataset = ConcatDataset([balanced_oulu, tapakah_dataset])

# Split 80/20
train_size = int(0.8 * len(combined_dataset))
val_size = len(combined_dataset) - train_size
train_dataset, val_dataset = random_split(combined_dataset, [train_size, val_size])

# Create dataloaders
BATCH_SIZE = 32
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

print(f"{'='*50}")
print("BALANCED DATASET CREATED")
print(f"{'='*50}")
print(f"Balanced OULU: {len(balanced_oulu)} samples")
print(f"Tapakah68: {len(tapakah_dataset)} samples")
print(f"Combined: {len(combined_dataset)} samples")
print(f"Train: {len(train_dataset)} | Val: {len(val_dataset)}")
print(f"{'='*50}")

BALANCED DATASET CREATED
Balanced OULU: 1372 samples
Tapakah68: 45 samples
Combined: 1417 samples
Train: 1133 | Val: 284


In [6]:
# ============================================
# CELL 6: TRAINING FUNCTIONS
# ============================================

def train_epoch(model, loader, criterion, optimizer, device):
    model.train()
    total_loss, correct, total = 0, 0, 0
    for images, labels in tqdm(loader, desc='Train'):
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        _, pred = outputs.max(1)
        correct += pred.eq(labels).sum().item()
        total += labels.size(0)
    return total_loss / len(loader), 100 * correct / total

def validate(model, loader, criterion, device):
    model.eval()
    total_loss, correct, total = 0, 0, 0
    with torch.no_grad():
        for images, labels in tqdm(loader, desc='Val'):
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            total_loss += loss.item()
            _, pred = outputs.max(1)
            correct += pred.eq(labels).sum().item()
            total += labels.size(0)
    return total_loss / len(loader), 100 * correct / total

print("✅ Training functions defined")

✅ Training functions defined


In [14]:
# ============================================
# CELL 11: EXTRACT MORE FRAMES FROM REPLAY VIDEOS
# ============================================
# The model is failing because we only have 9 replay videos.
# Let's extract 50 frames from each video to get more training data.

import os
import cv2
from pathlib import Path

REPLAY_DIR = Path('../data/anti-spoofing/replay')
EXTRACTED_DIR = Path('../data/extracted_replay_frames')
EXTRACTED_DIR.mkdir(exist_ok=True)

def extract_frames_from_video(video_path, output_dir, num_frames=50):
    """Extract multiple frames from a video."""
    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        return 0
    
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    if total_frames <= 0:
        cap.release()
        return 0
    
    # Get evenly spaced frame indices
    indices = np.linspace(0, total_frames - 1, num_frames, dtype=int)
    
    saved = 0
    video_name = video_path.stem
    for i, idx in enumerate(indices):
        cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
        ret, frame = cap.read()
        if ret:
            out_path = output_dir / f"{video_name}_frame{i:03d}.jpg"
            cv2.imwrite(str(out_path), frame)
            saved += 1
    
    cap.release()
    return saved

# Extract frames from all replay videos
print("Extracting frames from replay videos...")
total_extracted = 0
for video_file in REPLAY_DIR.glob('*'):
    if video_file.suffix.lower() in ['.mp4', '.mov', '.avi']:
        count = extract_frames_from_video(video_file, EXTRACTED_DIR, num_frames=50)
        print(f"  {video_file.name}: {count} frames")
        total_extracted += count

print(f"\n✅ Extracted {total_extracted} frames to {EXTRACTED_DIR}")
print("These are phone screen replay attack images (SPOOF)")


Extracting frames from replay videos...
  0001ffba3c--628c9dbb2579312f1ac7ee6f__-  20__- 22.mp4: 50 frames
  0001ffba3c--62960f430bd50b2755f74916__Galaxy M31__M2003J15SG.mp4: 50 frames
  0001ffba3c--628f8595ab45cb7cba7116fd__iPhone 12 PRO__iPhone X.MOV: 50 frames
  0001ffba3c--6295ba1f8753764e91dcf5df__Poco X3 Pro__Tecno Pouvoir 4.mp4: 50 frames
  0001ffba3c--628a68b5180bc205cf8704d2__Vivo y10__Honor 10 lite.mp4: 50 frames
  0001ffba3c--628e621c6789b1401e3cb184__Honor 50__Samsung note 9.mp4: 49 frames
  0001ffba3c--62880080572a894c5df8f427__Samsung Galaxy J2 Core __Samsung Galaxy A22.mp4: 50 frames
  0001ffba3c--6289196875a5cf63cd0cb302__Samsung galaxy s20 __IPhone 6s plus.mp4: 50 frames
  0001ffba3c--629066494796d7421dad66b7__iPhone xr __iPhone 6s.MOV: 50 frames

✅ Extracted 449 frames to ../data/extracted_replay_frames
These are phone screen replay attack images (SPOOF)


In [15]:
# ============================================
# CELL 12: CREATE ENHANCED DATASET WITH REPLAY FRAMES
# ============================================

class ReplayFramesDataset(Dataset):
    """Dataset for extracted replay frames (all are SPOOF)."""
    
    def __init__(self, frames_dir, transform=None):
        self.frames_dir = Path(frames_dir)
        self.transform = transform
        self.samples = []
        
        for f in self.frames_dir.glob('*.jpg'):
            self.samples.append((str(f), 0))  # label=0 for spoof
        
        print(f"Loaded {len(self.samples)} replay frames (all SPOOF)")
    
    def __len__(self):
        return len(self.samples)
    
    def __getitem__(self, idx):
        path, label = self.samples[idx]
        img = cv2.imread(path)
        if img is None:
            img = np.zeros((224, 224, 3), dtype=np.uint8)
        else:
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        if self.transform:
            img = self.transform(img)
        return img, label

# Create replay frames dataset
replay_dataset = ReplayFramesDataset(EXTRACTED_DIR, transform=train_transform)

# Combine: balanced OULU + Tapakah + extra replay frames
# We need to balance real vs spoof
# Real: from balanced_oulu (686) + tapakah real (18) = ~704
# Spoof: from balanced_oulu (686) + tapakah spoof (27) + replay frames (450) = ~1163

# Get real samples from combined dataset
combined_with_replay = ConcatDataset([balanced_oulu, tapakah_dataset, replay_dataset])

print(f"\n{'='*50}")
print("ENHANCED DATASET WITH REPLAY FRAMES")
print(f"{'='*50}")
print(f"Balanced OULU: {len(balanced_oulu)} samples")
print(f"Tapakah68: {len(tapakah_dataset)} samples")
print(f"Replay frames: {len(replay_dataset)} samples")
print(f"Total: {len(combined_with_replay)} samples")
print(f"{'='*50}")

# Split and create loaders
train_size2 = int(0.8 * len(combined_with_replay))
val_size2 = len(combined_with_replay) - train_size2
train_dataset2, val_dataset2 = random_split(combined_with_replay, [train_size2, val_size2])

train_loader2 = DataLoader(train_dataset2, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
val_loader2 = DataLoader(val_dataset2, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

print(f"Train: {len(train_dataset2)} | Val: {len(val_dataset2)}")


Loaded 449 replay frames (all SPOOF)

ENHANCED DATASET WITH REPLAY FRAMES
Balanced OULU: 1372 samples
Tapakah68: 45 samples
Replay frames: 449 samples
Total: 1866 samples
Train: 1492 | Val: 374


In [16]:
# ============================================
# CELL 13: RETRAIN WITH ENHANCED DATASET
# ============================================

# Create fresh model
model2 = LivenessDetector(pretrained=True)
model2.freeze_backbone()
model2 = model2.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model2.parameters(), lr=0.001)

print(f"{'='*50}")
print("STAGE 1: TRAINING WITH ENHANCED DATA (10 epochs)")
print(f"{'='*50}")

best_acc = 0
for epoch in range(10):
    print(f"\nEpoch {epoch+1}/10")
    train_loss, train_acc = train_epoch(model2, train_loader2, criterion, optimizer, device)
    val_loss, val_acc = validate(model2, val_loader2, criterion, device)
    print(f"Train: loss={train_loss:.4f}, acc={train_acc:.2f}%")
    print(f"Val:   loss={val_loss:.4f}, acc={val_acc:.2f}%")
    
    if val_acc > best_acc:
        best_acc = val_acc
        torch.save(model2.state_dict(), f'{MODELS_DIR}/liveness_enhanced_stage1.pth')
        print(f"✅ Saved best model (acc: {val_acc:.2f}%)")

print(f"\nStage 1 Complete! Best: {best_acc:.2f}%")


STAGE 1: TRAINING WITH ENHANCED DATA (10 epochs)

Epoch 1/10


Val: 100%|██████████| 12/12 [00:07<00:00,  1.67it/s]


Train: loss=0.4364, acc=79.69%
Val:   loss=0.3290, acc=87.43%
✅ Saved best model (acc: 87.43%)

Epoch 2/10


Val: 100%|██████████| 12/12 [00:07<00:00,  1.69it/s]


Train: loss=0.2960, acc=87.00%
Val:   loss=0.2777, acc=89.04%
✅ Saved best model (acc: 89.04%)

Epoch 3/10


...

Train: loss=0.2537, acc=89.81%
Val:   loss=0.2291, acc=93.32%
✅ Saved best model (acc: 93.32%)

Epoch 9/10


Val: 100%|██████████| 12/12 [00:07<00:00,  1.57it/s]


Train: loss=0.2593, acc=87.94%
Val:   loss=0.2850, acc=89.04%

Epoch 10/10


Val: 100%|██████████| 12/12 [00:08<00:00,  1.47it/s]

Train: loss=0.2478, acc=89.61%
Val:   loss=0.2191, acc=93.32%

Stage 1 Complete! Best: 93.32%


In [17]:
# ============================================
# CELL 14: FINE-TUNE ENHANCED MODEL
# ============================================

# Load best model and unfreeze
model2.load_state_dict(torch.load(f'{MODELS_DIR}/liveness_enhanced_stage1.pth'))
model2.unfreeze_backbone()

optimizer = optim.Adam(model2.parameters(), lr=0.0001)

print(f"{'='*50}")
print("STAGE 2: FINE-TUNING ENHANCED MODEL (5 epochs)")
print(f"{'='*50}")

best_acc = 0
for epoch in range(5):
    print(f"\nEpoch {epoch+1}/5")
    train_loss, train_acc = train_epoch(model2, train_loader2, criterion, optimizer, device)
    val_loss, val_acc = validate(model2, val_loader2, criterion, device)
    print(f"Train: loss={train_loss:.4f}, acc={train_acc:.2f}%")
    print(f"Val:   loss={val_loss:.4f}, acc={val_acc:.2f}%")
    
    if val_acc > best_acc:
        best_acc = val_acc
        torch.save(model2.state_dict(), f'{MODELS_DIR}/liveness_enhanced_final.pth')
        print(f"✅ Saved best model (acc: {val_acc:.2f}%)")

print(f"\nStage 2 Complete! Best: {best_acc:.2f}%")


STAGE 2: FINE-TUNING ENHANCED MODEL (5 epochs)

Epoch 1/5


Val: 100%|██████████| 12/12 [00:07<00:00,  1.54it/s]


Train: loss=0.1560, acc=93.90%
Val:   loss=0.1865, acc=94.65%
✅ Saved best model (acc: 94.65%)

Epoch 2/5


Val: 100%|██████████| 12/12 [00:07<00:00,  1.51it/s]


Train: loss=0.0756, acc=97.45%
Val:   loss=0.0662, acc=98.40%
✅ Saved best model (acc: 98.40%)

Epoch 3/5


...

Train: loss=0.0323, acc=98.99%
Val:   loss=0.0324, acc=98.66%
✅ Saved best model (acc: 98.66%)

Epoch 4/5


Val: 100%|██████████| 12/12 [00:09<00:00,  1.29it/s]


Train: loss=0.0327, acc=98.93%
Val:   loss=0.0766, acc=97.86%

Epoch 5/5


Val: 100%|██████████| 12/12 [00:08<00:00,  1.42it/s]

Train: loss=0.0306, acc=98.86%
Val:   loss=0.0280, acc=99.47%
✅ Saved best model (acc: 99.47%)

Stage 2 Complete! Best: 99.47%


In [18]:
# ============================================
# CELL 15: EXPORT ENHANCED MODEL & TEST
# ============================================

# Load best model
model2.load_state_dict(torch.load(f'{MODELS_DIR}/liveness_enhanced_final.pth'))
model2.eval()

# Save for production
torch.save({
    'model_state_dict': model2.state_dict(),
    'class_names': ['spoof', 'real'],
    'input_size': (3, 224, 224),
    'backbone': 'MobileNetV2',
    'trained_on': 'OULU-NPU + Tapakah68 + Replay Frames'
}, f'{MODELS_DIR}/liveness_detector_production.pth')

print(f"{'='*50}")
print("ENHANCED MODEL EXPORTED")
print(f"{'='*50}")
print(f"Saved to: {MODELS_DIR}/liveness_detector_production.pth")
print(f"{'='*50}")

# Update model reference for webcam test
model = model2
print("\n✅ Model updated. Run the webcam test cell again to test!")


ENHANCED MODEL EXPORTED
Saved to: ../models/liveness_detector_production.pth

✅ Model updated. Run the webcam test cell again to test!


In [20]:
# ============================================
# CELL 16: DOWNLOAD FACE DETECTOR
# ============================================
# The model was trained on CROPPED FACE images, not full frames!
# We need to detect face first, crop it, then run liveness.

import urllib.request
import os

os.makedirs('../models/face_detector', exist_ok=True)

prototxt_url = "https://raw.githubusercontent.com/opencv/opencv/master/samples/dnn/face_detector/deploy.prototxt"
caffemodel_url = "https://raw.githubusercontent.com/opencv/opencv_3rdparty/dnn_samples_face_detector_20170830/res10_300x300_ssd_iter_140000.caffemodel"

prototxt_path = '../models/face_detector/deploy.prototxt'
caffemodel_path = '../models/face_detector/res10_300x300_ssd_iter_140000.caffemodel'

if not os.path.exists(prototxt_path):
    print("Downloading face detector prototxt...")
    urllib.request.urlretrieve(prototxt_url, prototxt_path)

if not os.path.exists(caffemodel_path):
    print("Downloading face detector model (10MB)...")
    urllib.request.urlretrieve(caffemodel_url, caffemodel_path)

print("✅ Face detector files ready!")


✅ Face detector files ready!


In [22]:
# ============================================
# CELL 17: PROPER WEBCAM TEST WITH FACE DETECTION
# ============================================
# THIS IS THE KEY FIX: Detect face first, crop, then run liveness!

import cv2
import time
import numpy as np

# Load face detector
face_net = cv2.dnn.readNetFromCaffe(
    '../models/face_detector/deploy.prototxt',
    '../models/face_detector/res10_300x300_ssd_iter_140000.caffemodel'
)
print("✅ Face detector loaded")

def detect_face(frame, confidence_threshold=0.5):
    """Detect face using OpenCV DNN"""
    h, w = frame.shape[:2]
    blob = cv2.dnn.blobFromImage(frame, 1.0, (300, 300), (104.0, 177.0, 123.0))
    face_net.setInput(blob)
    detections = face_net.forward()
    
    for i in range(detections.shape[2]):
        confidence = detections[0, 0, i, 2]
        if confidence > confidence_threshold:
            box = detections[0, 0, i, 3:7] * np.array([w, h, w, h])
            x1, y1, x2, y2 = box.astype(int)
            return (x1, y1, x2, y2), confidence
    return None, 0

def test_with_face_detection(model, device, duration=30):
    """
    PROPER liveness detection:
    1. Detect face
    2. Crop face region  
    3. Run liveness on CROPPED FACE only
    """
    cap = cv2.VideoCapture(0)
    if not cap.isOpened():
        print("❌ Could not open webcam")
        return
    
    print("=" * 60)
    print("LIVENESS TEST WITH FACE DETECTION (PROPER METHOD)")
    print("=" * 60)
    print("Pipeline: Frame → Detect Face → Crop → Liveness Model")
    print(f"Running for {duration} seconds... Press 'q' to quit")
    print("=" * 60)
    
    model.eval()
    start = time.time()
    
    while time.time() - start < duration:
        ret, frame = cap.read()
        if not ret:
            break
        
        # Step 1: Detect face
        face_box, face_conf = detect_face(frame)
        
        if face_box is not None:
            x1, y1, x2, y2 = face_box
            
            # Add 20% padding
            h, w = frame.shape[:2]
            pad_x = int((x2 - x1) * 0.2)
            pad_y = int((y2 - y1) * 0.2)
            x1 = max(0, x1 - pad_x)
            y1 = max(0, y1 - pad_y)
            x2 = min(w, x2 + pad_x)
            y2 = min(h, y2 + pad_y)
            
            # Step 2: Crop face (RGB)
            face_crop = frame[y1:y2, x1:x2]
            if face_crop.size > 0:
                face_rgb = cv2.cvtColor(face_crop, cv2.COLOR_BGR2RGB)
                
                # Step 3: Run liveness on CROPPED FACE
                tensor = val_transform(face_rgb).unsqueeze(0).to(device)
                
                with torch.no_grad():
                    out = model(tensor)
                    probs = torch.softmax(out, dim=1)
                    spoof_prob = probs[0][0].item()
                    real_prob = probs[0][1].item()
                
                # Display results
                is_real = real_prob > 0.5
                label = "REAL" if is_real else "SPOOF"
                color = (0, 255, 0) if is_real else (0, 0, 255)
                
                # Draw face box
                cv2.rectangle(frame, (x1, y1), (x2, y2), color, 3)
                
                # Draw label above face
                cv2.putText(frame, f"{label}: {max(real_prob, spoof_prob)*100:.1f}%", 
                            (x1, y1-10), cv2.FONT_HERSHEY_SIMPLEX, 1, color, 2)
                
                # Show probabilities
                cv2.putText(frame, f"Real: {real_prob*100:.1f}%", (10, 30),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)
                cv2.putText(frame, f"Spoof: {spoof_prob*100:.1f}%", (10, 60),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)
        else:
            cv2.putText(frame, "No face detected", (10, 30), 
                        cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 165, 255), 2)
        
        cv2.putText(frame, "FACE CROP + LIVENESS", (10, frame.shape[0]-10),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 255), 1)
        
        cv2.imshow('Liveness Detection (Face Crop)', frame)
        
        if cv2.waitKey(1) & 0xFF == ord('q'):
            print("\n⏹️ Stopped by user")
            break
    
    cap.release()
    cv2.destroyAllWindows()
    print("✅ Test complete!")

# Run the proper test
test_with_face_detection(model, device, duration=30)

✅ Face detector loaded
LIVENESS TEST WITH FACE DETECTION (PROPER METHOD)
Pipeline: Frame → Detect Face → Crop → Liveness Model
Running for 30 seconds... Press 'q' to quit
✅ Test complete!


---
## ✅ Training Complete!

The model is saved at: `../models/liveness_detector_production.pth`

This model is ready for backend integration.